# RAG Teaching Workbench

Use this notebook to:
- provide or edit `docs.txt` content,
- rebuild the FAISS index,
- run queries and inspect retrieved chunks + citations,
- do quick quality checks.

## 1) Setup imports and paths

In [ ]:
from pathlib import Path
import os
import sys
import textwrap

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from main import RAGPipeline

print('Project root:', PROJECT_ROOT)
print('Src dir:', SRC_DIR)

## 2) Optional runtime knobs
Set any environment variable here before creating `RAGPipeline()`.

In [ ]:
os.environ['RETRIEVAL_K'] = '3'
os.environ['FETCH_K'] = '10'
os.environ['MIN_RELEVANCE_SCORE'] = '0.35'
os.environ['MAX_CONTEXT_CHARS'] = '2500'
os.environ['STEP_BY_STEP_MODE'] = 'false'

# Uncomment if needed
# os.environ['EMB_MODEL'] = str(PROJECT_ROOT / 'models' / 'all-MiniLM-L6-v2')
# os.environ['GEN_MODEL'] = str(PROJECT_ROOT / 'models' / 't5-small')


## 3) Create pipeline

In [ ]:
rag = RAGPipeline()
rag.config

## 4) Provide your teaching documents
Write one long or short document per line in `docs.txt`.

In [ ]:
sample_docs = [
    'RAG combines retrieval with generation to improve factual grounding.',
    'Retriever converts text to embeddings and searches FAISS for relevant chunks.',
    'Generator answers using only retrieved context and should cite chunk ids.',
    'Chunking large documents improves retrieval precision for specific questions.'
]

docs_file = rag.config.docs_file
docs_file.parent.mkdir(parents=True, exist_ok=True)
docs_file.write_text('\n'.join(doc.strip() for doc in sample_docs if doc.strip()) + '\n', encoding='utf-8')
print('Wrote docs to:', docs_file)
print('Document lines:', len(sample_docs))

## 5) Build or rebuild index

In [ ]:
num_chunks = rag.retriever.build_index()
print('Indexed chunks:', num_chunks)
print('Index file:', rag.config.index_file)
print('Meta file:', rag.config.meta_file)

## 6) Ask one question

In [ ]:
query = 'How does chunking help RAG?'
result = rag.run(query, k=rag.config.retrieval_k)

print('Question:', result['query'])
print('Citations:', result['citations'])
print('Answer:\n', result['answer'])
print('\nRetrieved chunks:')
for i, chunk in enumerate(result['retrieved_chunks'], 1):
    preview = textwrap.shorten(chunk['text'], width=120, placeholder='...')
    print(f"{i}. chunk={chunk['chunk_id']} source={chunk['source_doc_id']} score={chunk['score']:.3f} | {preview}")

## 7) Batch testing with quick checks

In [ ]:
test_queries = [
    'What is RAG?',
    'What does retriever do?',
    'Why is chunking useful?'
]

for q in test_queries:
    out = rag.run(q, k=rag.config.retrieval_k)
    print('\n' + '=' * 60)
    print('Q:', q)
    print('Answer:', out['answer'])
    print('Citations:', out['citations'])
    print('Retrieved:', len(out['retrieved_chunks']))

    # Lightweight teaching checks
    assert isinstance(out['retrieved_chunks'], list)
    assert isinstance(out['citations'], list)
    if out['retrieved_chunks']:
        valid_ids = {c['chunk_id'] for c in out['retrieved_chunks']}
        assert set(out['citations']).issubset(valid_ids)

print('\nQuick checks passed.')

## 8) Helper: inspect current docs.txt quickly

In [ ]:
docs_text = rag.config.docs_file.read_text(encoding='utf-8')
lines = [line for line in docs_text.splitlines() if line.strip()]
print('docs.txt lines:', len(lines))
print('First line preview:', textwrap.shorten(lines[0], width=160, placeholder='...') if lines else '(empty)')